<a href="https://colab.research.google.com/github/dewi-ops/Data-Analytic-and-Infrastucture/blob/main/DAI_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()


Saving Sample - Superstore.csv to Sample - Superstore (1).csv


In [3]:
import pandas as pd
df = pd.read_csv("Sample - Superstore.csv", encoding='latin1')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [4]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
dim_date = df[["Order Date", "Year", "Month"]].drop_duplicates()
dim_date["date_id"] = range(1, len(dim_date) + 1)
dim_date.head()

,Order Date,Year,Month,date_id
0,2016-11-08,2016,11,1
2,2016-06-12,2016,6,2
3,2015-10-11,2015,10,3
5,2014-06-09,2014,6,4
12,2017-04-15,2017,4,5


In [5]:
dim_product = df[["Category", "Sub-Category"]].drop_duplicates()
dim_product["product_id"] = range(1, len(dim_product) + 1)
dim_product.head()

,Category,Sub-Category,product_id
0,Furniture,Bookcases,1
1,Furniture,Chairs,2
2,Office Supplies,Labels,3
3,Furniture,Tables,4
4,Office Supplies,Storage,5


In [6]:
dim_region = df[["Region"]].drop_duplicates()
dim_region["region_id"] = range(1, len(dim_region) + 1)
dim_region.head()


,Region,region_id
0,South,1
2,West,2
14,Central,3
23,East,4


In [7]:
fact_table = df.merge(dim_date, on=["Order Date", "Year", "Month"], how='left') \
               .merge(dim_product, on=["Category", "Sub-Category"], how='left') \
               .merge(dim_region, on=["Region"], how='left')
fact_table.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Product Name,Sales,Quantity,Discount,Profit,Year,Month,date_id,product_id,region_id
0,1,CA-2016-152156,2016-11-08,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,11,1,1,1
1,2,CA-2016-152156,2016-11-08,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,11,1,2,1
2,3,CA-2016-138688,2016-06-12,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,6,2,3,2
3,4,US-2015-108966,2015-10-11,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,10,3,4,1
4,5,US-2015-108966,2015-10-11,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,10,3,5,1


In [8]:
fact_table['Profit_Margin'] = fact_table['Profit'] / fact_table['Sales']
fact_sales = fact_table[[
 "date_id",
 "product_id",
 "region_id",
 "Sales",
 "Profit",
 "Profit_Margin"
]]
fact_sales.head()

,date_id,product_id,region_id,Sales,Profit,Profit_Margin
0,1,1,1,261.9600,41.9136,0.1600
1,1,2,1,731.9400,219.5820,0.3000
2,2,3,2,14.6200,6.8714,0.4700
3,3,4,1,957.5775,-383.0310,-0.4000
4,3,5,1,22.3680,2.5164,0.1125


In [9]:
import os
os.makedirs("data/warehouse", exist_ok=True)
dim_date.to_csv("data/warehouse/dim_date.csv", index=False)
dim_product.to_csv("data/warehouse/dim_product.csv", index=False)
dim_region.to_csv("data/warehouse/dim_region.csv", index=False)
fact_sales.to_csv("data/warehouse/fact_sales.csv", index=False)

In [10]:
ales_mart = fact_sales.merge(dim_date, on="date_id") \
 .merge(dim_product, on="product_id") \
 .merge(dim_region, on="region_id")
ales_mart.head()

,date_id,product_id,region_id,Sales,Profit,Profit_Margin,Order Date,Year,Month,Category,Sub-Category,Region
0,1,1,1,261.9600,41.9136,0.1600,2016-11-08,2016,11,Furniture,Bookcases,South
1,1,2,1,731.9400,219.5820,0.3000,2016-11-08,2016,11,Furniture,Chairs,South
2,2,3,2,14.6200,6.8714,0.4700,2016-06-12,2016,6,Office Supplies,Labels,West
3,3,4,1,957.5775,-383.0310,-0.4000,2015-10-11,2015,10,Furniture,Tables,South
4,3,5,1,22.3680,2.5164,0.1125,2015-10-11,2015,10,Office Supplies,Storage,South
